# Modelo de Identificação de Fraudes

In [1]:
# Importando o pandas

import pandas as pd

In [2]:
# Importando a base

transacoes = pd.read_csv('creditcard.csv')

<a id='modelo'></a>
### Criando um modelo de aprendizado de máquinas para identificar fraude
- Vamos criar um modelo para identificar fraude sem fazer nenhum tratamento nessa base

**Podemos ajustar as colunas Time e Amount para que elas fiquem entre 0 e 1**

In [3]:
# Primeiro para a coluna Time

transacoes.Time = transacoes.Time / transacoes.Time.max()

In [4]:
# E então para a coluna Amount

transacoes.Amount = transacoes.Amount / transacoes.Amount.max()

In [5]:
# Separando X e y

X = transacoes.drop('Class',axis=1)
y = transacoes.Class

**Separando em treino e teste**

In [6]:
# Separando em treino e teste

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=0,stratify=y)

In [7]:
# Contando os valores

y_train.value_counts()

0    190490
1       330
Name: Class, dtype: int64

### Realizando o random undersampling

In [8]:
# Importando o RandomUnderSampler do imblearn

from imblearn.under_sampling import RandomUnderSampler 

In [9]:
# Definindo o RandomUnderSampler

rus = RandomUnderSampler(random_state=42)

In [10]:
# Definindo a nova amostra

X_resRUS, y_resRUS = rus.fit_resample(X_train, y_train)

In [11]:
# Contando os valores após equilibrar o dataset

y_resRUS.value_counts()

0    330
1    330
Name: Class, dtype: int64

### Nos passos anteriores, tivemos como os melhores hiperparâmetros:
- **<font color='blue'>Regressão Logística:**
    - {'C': 0.001, 'solver': 'liblinear'}\
- **<font color='blue'>SVC:**
    - {'C': 100, 'gamma': 'auto', 'kernel': 'rbf'}
- **<font color='blue'>Random Forest:**
    - {'criterion': 'gini', 'max_depth': 9, 'n_estimators': 30}

In [12]:
# Importando a matriz de confusão

from sklearn.metrics import confusion_matrix

In [13]:
# Regressão Logística

from sklearn.linear_model import LogisticRegression

clf_RL = LogisticRegression(random_state=0,
                             C=0.001,
                             solver='liblinear').fit(X_resRUS, y_resRUS)

y_pred_RL = clf_RL.predict(X_test)

confusion_matrix(y_test,y_pred_RL)

array([[85217,  8608],
       [   10,   152]], dtype=int64)

In [14]:
# SVC

from sklearn.svm import SVC

clf_SVC = SVC(random_state=0,
              probability=True,
             C=100,gamma='auto',
             kernel='rbf').fit(X_resRUS, y_resRUS)

y_pred_SVC = clf_SVC.predict(X_test)

confusion_matrix(y_test,y_pred_SVC)

array([[85791,  8034],
       [   12,   150]], dtype=int64)

In [15]:
# Random Forest

from sklearn.ensemble import RandomForestClassifier

clf_RF = RandomForestClassifier(max_depth=9, 
                                random_state=0,
                                criterion='gini',
                                n_estimators=30).fit(X_resRUS, y_resRUS)

y_pred_RF = clf_RF.predict(X_test)

confusion_matrix(y_test,y_pred_RF)

array([[91476,  2349],
       [   19,   143]], dtype=int64)

### Podemos começar adicionando novos parâmetros

**E novamente utilizar o GridSearchCV**

In [16]:
# Importando o GridSearchCV

from sklearn.model_selection import GridSearchCV

In [17]:
# Adicionando novos parâmetros

parametros = {
    'C': [0.001,0.01,0.1,1,10,100,1000],
    'solver': ['newton-cg','lbfgs','liblinear','sag','saga'],
    'penalty': ['l1', 'l2', 'elasticnet', None],
    'max_iter': [100,500,1000]
}

In [18]:
# Selecionando a Regressão Logística

LogReg = LogisticRegression(random_state=42)

**Retirando os warnings**

In [19]:
import warnings

warnings.filterwarnings('ignore')

In [20]:
# Criando o classificador

clf_RL2 = GridSearchCV(LogReg, parametros,
                       scoring='recall'
                      ).fit(X_resRUS,y_resRUS)

In [21]:
# E avaliando o modelo

y_pred_RL2 = clf_RL2.predict(X_test)

confusion_matrix(y_test,y_pred_RL2)

array([[78529, 15296],
       [   11,   151]], dtype=int64)

In [22]:
# Verificando o recall apenas com C e solver

from sklearn.metrics import recall_score

# recall_score(y_test, y_pred_RL2)

# 0.9382716049382716

In [23]:
# Verificando o recall com os novos parâmetros

from sklearn.metrics import recall_score

recall_score(y_test, y_pred_RL2)

0.9320987654320988

### Podemos adicionar novas métricas no scoring (como, por exemplo, adicionar a precisão além do recall)

In [32]:
# Criando o classificador

clf_RL3 = GridSearchCV(LogReg, parametros,
                       scoring=['recall', 'precision'],
                       refit='recall'
                      ).fit(X_resRUS,y_resRUS)

In [33]:
# Avaliando o novo modelo

y_pred_RL3 = clf_RL3.predict(X_test)

confusion_matrix(y_test,y_pred_RL3)

array([[78529, 15296],
       [   11,   151]], dtype=int64)